# M1 lab: from dispatch to locational prices

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M1 · about 45 minutes, plus 90 minutes of practice at the end.

You will build the lesson's linear program in Python, solve it with the open-source HiGHS solver through `scipy.optimize.linprog(method="highs")`, read every price as the dual of a balance row, and check with `assert` cells that you get the same numbers as the browser.

**How to run it.** In Google Colab: *File → Upload notebook*, choose this file, then *Runtime → Run all* (Colab already has NumPy and SciPy). Locally: open it in JupyterLab, Jupyter Notebook or VS Code (with its Jupyter extension), with Python 3.9 or later, `numpy` and `scipy` (1.9 or later); `matplotlib` is optional, for the one plot. No other files are needed: the data are in the notebook.

**The data.** Prairie Town is fictional and its plants, costs, network and line limits are illustrative. Demand, wind and solar follow the real hourly shapes of 3 September 2025 on the Southwest Power Pool grid (EIA Form EIA-930, Hourly Electric Grid Monitor), scaled to the town exactly as in lesson 0.

In [ ]:
import numpy as np
import scipy
from scipy.optimize import linprog

print("NumPy", np.__version__, "· SciPy", scipy.__version__)
major, minor = (int(x) for x in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this lab needs SciPy 1.9 or later (pip install -U scipy)"

## 1. The day and the plants

Every number below is the same as in the browser lesson. Hours run 0 to 23; each lasts one hour, so a plant's output in MW over the hour is also its energy in MWh.

In [ ]:
HOURS = 24
demand = [250, 243, 238, 235, 236, 237, 253, 260, 266, 278, 286, 300, 311, 327, 340, 344, 353, 349, 344, 331, 319, 303, 285, 269]
wind   = [30, 28, 28, 30, 36, 42, 43, 48, 46, 35, 36, 38, 35, 33, 33, 37, 47, 54, 59, 72, 89, 98, 95, 89]
solar  = [0, 0, 0, 0, 0, 0, 0, 7, 70, 118, 130, 147, 156, 149, 112, 147, 161, 130, 70, 14, 0, 0, 0, 0]

# plant: size (MW), running cost ($/MWh), bus. The nuclear plant runs flat at 100 MW, as in lesson 0.
PLANTS = {
    "nuclear": dict(size=100, cost=10,  bus="T"),
    "wind":    dict(size=150, cost=0,   bus="R"),
    "solar":   dict(size=200, cost=0,   bus="R"),
    "cc":      dict(size=150, cost=40,  bus="G"),   # gas combined-cycle: ramps at most 40 MW per hour, starts from 115 MW
    "pk":      dict(size=100, cost=100, bus="G"),   # gas peaker
}
RAMP, CC_START = 40, 115
BUSES = ["T", "R", "G"]                 # Prairie Town (the reference bus: angle 0, all the demand), Wind Ridge, Gas Hub
LINES = {"RT": ("R", "T"), "RG": ("R", "G"), "GT": ("G", "T")}
B = 1000.0                              # line susceptance, MW per radian, equal on all three lines

def available(g, t):
    return wind[t] if g == "wind" else solar[t] if g == "solar" else PLANTS[g]["size"]

assert len(demand) == len(wind) == len(solar) == HOURS
print(f"Peak demand {max(demand)} MW at {demand.index(max(demand)):02d}:00")

## 2. The model as a linear program

`linprog` solves: minimise $c^\top x$ subject to $A_{ub}\,x \le b_{ub}$, $A_{eq}\,x = b_{eq}$ and bounds on $x$.

| Rule | In symbols | Rows |
|---|---|---|
| Goal | $\min \sum_t \sum_g c_g\, p_{g,t}$ | the objective, not a row |
| Balance, one bus | $\sum_g p_{g,t} = d_t$ | 24 equality rows |
| Balance, network | $\sum_{g \in n} p_{g,t} + \text{inflow}_{n,t} - \text{outflow}_{n,t} = d_{n,t}$ | 3 × 24 equality rows |
| Plant sizes, wind and solar, nuclear flat | $0 \le p_{g,t} \le \bar P_g$ (wind, solar: $\le A_{g,t}$; nuclear $= 100$) | bounds |
| Ramp | $-40 \le p_{\text{cc},t} - p_{\text{cc},t-1} \le 40$, with $p_{\text{cc},-1} = 115$ | 2 × 24 inequality rows |
| DC power flow | $f_{\ell,t} = b_\ell\,(\theta_{i,t} - \theta_{j,t})$, $\theta_{\text{Town},t} = 0$ | 3 × 24 equality rows |
| Line limits | $-F_\ell \le f_{\ell,t} \le F_\ell$ | bounds on the flows |

Symbols: $t$ hour (each lasts 1 h, so MW over the hour = MWh); $g$ plant; $n$ bus; $\ell$ line, with end buses $i$ and $j$; $c_g$ running cost, \$/MWh; $\bar P_g$ plant size, MW; $A_{g,t}$ what the weather gives, MW; $d_{n,t}$ demand, MW; $f_{\ell,t}$ flow, MW; $\theta_{n,t}$ voltage angle, radians; $b_\ell$ susceptance, which turns an angle difference into flow (MW per radian); $F_\ell$ line limit, MW; $p_{\text{cc},-1}$ the gas plant's output before the day starts.

The function below writes these rows. The line limits are written as two inequality rows each, not as bounds, so that their duals are easy to find.

In [ ]:
def build(network=False, limit_RT=200.0, other_limits=200.0, ramp=True, add=None, limits=None):
    # Returns the linprog inputs and an index of where each variable and row sits.
    # limits = {line: MW} sets the limits of the lines it names (it overrides limit_RT and other_limits).
    # add = (bus, hour, MWh) adds demand at one bus and hour (to check a price by a finite difference).
    names = []
    for t in range(HOURS):
        names += [f"{g}_{t}" for g in PLANTS]
        if network:
            names += [f"f{l}_{t}" for l in LINES] + [f"th{n}_{t}" for n in BUSES if n != "T"]
    col = {name: j for j, name in enumerate(names)}
    nv = len(names)
    c = np.zeros(nv); bounds = [(0, None)] * nv
    for t in range(HOURS):
        for g, p in PLANTS.items():
            c[col[f"{g}_{t}"]] = p["cost"]
            hi = available(g, t)
            bounds[col[f"{g}_{t}"]] = (hi, hi) if g == "nuclear" else (0, hi)
        if network:
            for l in LINES:
                bounds[col[f"f{l}_{t}"]] = (None, None)
            for n in BUSES:
                if n != "T":
                    bounds[col[f"th{n}_{t}"]] = (None, None)
    A_eq, b_eq, eq_rows = [], [], []
    A_ub, b_ub, ub_rows = [], [], []
    def row(terms):
        r = np.zeros(nv)
        for name, v in terms.items():
            r[col[name]] += v
        return r
    for t in range(HOURS):
        if network:
            for n in BUSES:
                terms = {f"{g}_{t}": 1 for g, p in PLANTS.items() if p["bus"] == n}
                for l, (a, b) in LINES.items():
                    if a == n: terms[f"f{l}_{t}"] = -1      # power leaving n
                    if b == n: terms[f"f{l}_{t}"] = 1       # power arriving at n
                d = demand[t] if n == "T" else 0
                if add and add[0] == n and add[1] == t: d += add[2]
                A_eq.append(row(terms)); b_eq.append(d); eq_rows.append(f"bal_{n}_{t}")
            for l, (a, b) in LINES.items():
                terms = {f"f{l}_{t}": 1}
                if a != "T": terms[f"th{a}_{t}"] = -B
                if b != "T": terms[f"th{b}_{t}"] = B
                A_eq.append(row(terms)); b_eq.append(0); eq_rows.append(f"flow_{l}_{t}")
                F = limit_RT if l == "RT" else other_limits
                if limits and l in limits: F = limits[l]
                A_ub.append(row({f"f{l}_{t}": 1}));  b_ub.append(F); ub_rows.append(f"cap+_{l}_{t}")
                A_ub.append(row({f"f{l}_{t}": -1})); b_ub.append(F); ub_rows.append(f"cap-_{l}_{t}")
        else:
            d = demand[t] + (add[2] if add and add[1] == t else 0)
            A_eq.append(row({f"{g}_{t}": 1 for g in PLANTS})); b_eq.append(d); eq_rows.append(f"bal_{t}")
        if ramp:
            prev = {f"cc_{t - 1}": -1} if t > 0 else {}
            up, lo = (RAMP, RAMP) if t > 0 else (CC_START + RAMP, RAMP - CC_START)
            A_ub.append(row({f"cc_{t}": 1, **prev})); b_ub.append(up); ub_rows.append(f"ramp+_{t}")
            A_ub.append(row({f"cc_{t}": -1, **{k: -v for k, v in prev.items()}})); b_ub.append(lo); ub_rows.append(f"ramp-_{t}")
    lp = dict(c=c, A_eq=np.array(A_eq), b_eq=np.array(b_eq), A_ub=np.array(A_ub) if A_ub else None,
              b_ub=np.array(b_ub) if b_ub else None, bounds=bounds)
    return lp, col, eq_rows, ub_rows

def solve(**kw):
    lp, col, eq_rows, ub_rows = build(**kw)
    res = linprog(method="highs", **lp)
    assert res.status == 0, res.message
    out = dict(res=res, cost=res.fun, x={name: res.x[j] for name, j in col.items()})
    # eqlin.marginals = d(optimal cost) / d(right-hand side): for a balance row, the cost of one more MWh there.
    out["dual"] = dict(zip(eq_rows, res.eqlin.marginals))
    out["ub_dual"] = dict(zip(ub_rows, res.ineqlin.marginals)) if ub_rows else {}
    return out

lp, col, eq_rows, ub_rows = build()
print(len(col), "variables,", len(eq_rows) + len(ub_rows), "rows (balance rows and the ramp rule's two-sided rows)")

## 3. Part A: the day on one bus

Solve, then read the 24 balance duals. They are the hourly prices of the lesson: local marginal values (the extra cost per MWh of a small change in demand, away from a corner of the model) under stated assumptions (fixed plants, costs used as offers, the modelled day), not a market's settlement.

In [ ]:
A = solve()
price_A = [A["dual"][f"bal_{t}"] for t in range(HOURS)]
print(f"Cost of the day: ${A['cost']:,.0f}")
print("Hourly prices ($/MWh):", [round(p) for p in price_A])

# the browser's numbers
assert abs(A["cost"] - 104000) < 1e-6
browser_A = [40, 40, 40, 40, 40, 40, 40, 80, 0, 40, 40, 40, 40, 0, 80, 40, 40, 0, 80, 40, 40, 40, 40, 40]
assert np.allclose(price_A, browser_A, atol=1e-6), "prices differ from the browser"
print("Part A matches the browser.")

**Check a price by a finite difference.** Add 1 MWh of demand at 19:00 and solve again: the cost should rise by the dual. Then do the same at 14:00 and look at which outputs change.

In [ ]:
A19 = solve(add=("T", 19, 1.0))
print(f"+1 MWh at 19:00: cost rises by ${A19['cost'] - A['cost']:.2f}")
assert abs((A19["cost"] - A["cost"]) - 40) < 1e-6

eps = 0.1
A14 = solve(add=("T", 14, eps))
changes = {k: (A14["x"][k] - A["x"][k]) / eps for k in A["x"] if abs(A14["x"][k] - A["x"][k]) > 1e-9}
print("Per MWh added at 14:00:", {k: round(float(v), 3) for k, v in changes.items()})
print(f"Cost per MWh: ${(A14['cost'] - A['cost']) / eps:.2f}")
assert abs((A14["cost"] - A["cost"]) / eps - 80) < 1e-6
assert abs(changes["cc_13"] - 1) < 1e-6 and abs(changes["cc_14"] - 1) < 1e-6

**Why $80?** The gas plant can climb only 40 MW an hour, and that rule binds between 13:00 and 14:00. One more MWh at 14:00 needs the plant 1 MWh higher at 14:00 *and* at 13:00, where the extra MWh only pushes out free wind and solar: $40 + $40 = $80. Now switch the ramp rule off.

In [ ]:
A_off = solve(ramp=False)
price_off = [A_off["dual"][f"bal_{t}"] for t in range(HOURS)]
print(f"Without the ramp rule: ${A_off['cost']:,.0f}; prices {sorted(set(round(p) for p in price_off))}")
assert abs(A_off["cost"] - 102600) < 1e-6 and np.allclose(price_off, 40)
print(f"The ramp rule costs ${A['cost'] - A_off['cost']:,.0f} on this day.")

## 4. Part B: three buses and DC power flow

Wind and solar sit on Wind Ridge (R), both gas plants at the Gas Hub (G), the nuclear plant and all the demand in Prairie Town (T). Each bus has its own balance row, so each bus has its own dual: the **locational marginal price**.

First, with every line at 200 MW nothing fills, and the network must give back part A's prices at every bus.

In [ ]:
def lmps(sol):
    return {n: [sol["dual"][f"bal_{n}_{t}"] for t in range(HOURS)] for n in BUSES}

B200 = solve(network=True, limit_RT=200)
p200 = lmps(B200)
assert abs(B200["cost"] - 104000) < 1e-6
assert all(np.allclose(p200[n], browser_A, atol=1e-6) for n in BUSES)
busiest = max(abs(B200["x"][f"f{l}_{t}"]) for l in LINES for t in range(HOURS))
print(f"200 MW lines: same cost and prices as part A; busiest line {busiest:.1f} MW")

Now tighten the Ridge–Town line to 110 MW, as in the lesson's example.

In [ ]:
B110 = solve(network=True, limit_RT=110)
p110 = lmps(B110)
full = [t for t in range(HOURS) if abs(B110["x"][f"fRT_{t}"]) >= 110 - 1e-6]
print(f"Cost ${B110['cost']:,.0f}; Ridge–Town full in hours {full}")
for t in (12, 19):
    print(f"{t:02d}:00  Ridge ${p110['R'][t] + 0:.0f}  Gas Hub ${p110['G'][t] + 0:.0f}  Town ${p110['T'][t] + 0:.0f}   flow RT {B110['x'][f'fRT_{t}']:.1f} MW")

# the browser's numbers
assert abs(B110["cost"] - 133160) < 1e-6
assert np.allclose([p110["R"][12], p110["G"][12], p110["T"][12]], [0, 40, 80], atol=1e-6)
assert full == list(range(9, 19))
assert np.allclose([p110[n][19] for n in BUSES], 40) and round(B110["x"]["fRT_19"]) == 106
print("Part B matches the browser.")

**Explain the town's price in words.** Add a little demand in town at noon and see which outputs move.

In [ ]:
Bt = solve(network=True, limit_RT=110, add=("T", 12, eps))
moves = {k: (Bt["x"][k] - B110["x"][k]) / eps for k in B110["x"]
         if k.split("_")[0] in PLANTS and abs(Bt["x"][k] - B110["x"][k]) > 1e-9}
print("Per MWh added in town at noon:", {k: round(float(v), 3) for k, v in moves.items()})
ren = sum(v for k, v in moves.items() if k.startswith(("wind", "solar")))
assert abs(moves["cc_12"] - 2) < 1e-6 and abs(ren + 1) < 1e-6
print("Gas Hub +2 MWh, Ridge -1 MWh: 2 x $40 - 1 x $0 = $80. The two changes cancel on the full line.")

## 5. Two identities to check

**Prices from PTDFs.** With the town as reference bus, a power transfer distribution factor $\text{PTDF}_{\ell n}$ is the share of 1 MW injected at bus $n$ (and taken out in town) that flows on line $\ell$. In this lossless model
$$\text{LMP}_n = \text{LMP}_\text{Town} - \sum_\ell \text{PTDF}_{\ell n}\,\mu_\ell ,$$
where $\mu_\ell$ is the value of 1 MW more limit on line $\ell$ (minus the dual of its upper limit row).

**Congestion rent.** What loads pay minus what generators are paid, each at its own bus's price, equals $\sum_\ell \mu_\ell\, f_\ell$ in this lossless DC model. It is not the price gap between one line's ends times that line's flow.

In [ ]:
# PTDFs of the triangle (equal susceptances): from the reduced susceptance matrix of buses R and G
Bred = np.array([[2 * B, -B], [-B, 2 * B]])
X = np.linalg.inv(Bred)
idx = {"R": 0, "G": 1}
def ptdf(l, n):
    a, b = LINES[l]
    if n == "T": return 0.0
    th = lambda k: X[idx[k], idx[n]] if k != "T" else 0.0
    return B * (th(a) - th(b))
P = {l: {n: ptdf(l, n) for n in BUSES} for l in LINES}
print("PTDF (line: {bus: share}):", {l: {n: round(float(v), 4) for n, v in P[l].items()} for l in LINES})

t = 12
mu = {l: -B110["ub_dual"][f"cap+_{l}_{t}"] + B110["ub_dual"][f"cap-_{l}_{t}"] for l in LINES}
print("mu at noon ($ per MW of limit):", {l: round(float(v), 2) + 0 for l, v in mu.items()})
for n in BUSES:
    rebuilt = p110["T"][t] - sum(P[l][n] * mu[l] for l in LINES)
    assert abs(rebuilt - p110[n][t]) < 1e-6
inj = {n: sum(B110["x"][f"{g}_{t}"] for g, p in PLANTS.items() if p["bus"] == n) for n in BUSES}
paid = sum(p110[n][t] * ((demand[t] if n == "T" else 0) - inj[n]) for n in BUSES)
via_mu = sum(mu[l] * B110["x"][f"f{l}_{t}"] for l in LINES)
print(f"Congestion rent at noon: ${paid:,.0f} = sum of mu x flow ${via_mu:,.0f}")
assert abs(mu["RT"] - 120) < 1e-6 and abs(paid - 13200) < 1e-6 and abs(paid - via_mu) < 1e-6

In [ ]:
# Optional: the 24 prices at the three buses (needs matplotlib)
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed: skipping the plot")
if plt is not None:
    fig, ax = plt.subplots(figsize=(8, 3))
    for n, label in (("T", "Prairie Town"), ("R", "Wind Ridge"), ("G", "Gas Hub")):
        ax.step(range(HOURS), p110[n], where="post", label=label)
    ax.set_xlabel("Hour"); ax.set_ylabel("$ per MWh"); ax.set_title("Prices with the Ridge–Town line at 110 MW")
    ax.legend(); plt.show()

## 6. Practice (about 90 minutes)

Write your own cells below each question. Predict first, then solve. The last cell checks your answers.

1. **Find the edge.** Lower the Ridge–Town limit below 90 MW in steps of 1 MW. At what limit does the model become infeasible, and which hour fails first? Explain why with the sizes of the two gas plants.
2. **Move a plant.** Put the nuclear plant on Wind Ridge instead of in town (`PLANTS["nuclear"]["bus"] = "R"`).
   - At 110 MW the day has no feasible plan. Show it, and explain it with the town's noon demand and the limits of its two lines.
   - Find the lowest whole-MW Ridge–Town limit that solves. Which hour sets it?
   - At 160 MW, predict the noon prices. Then solve and explain each price with a finite difference, as in section 4.
   - Put the plant back in town (`"T"`) before the next task.
3. **The third line.** Limit the Ridge–Gas Hub line to 40 MW (`limits={"RG": 40}`) and leave the other two at 200 MW. Which hours congest, and in what order are the three prices now? Explain the order with the PTDFs of section 5, and the town's price with a finite difference.
4. **The PTDF form.** Rewrite the network part without angles: flows $f = \text{PTDF} \cdot$ (net injections). Confirm that you get the same prices as the angle form.
5. **A price is not always unique.** At 14:00 with the line at 110 MW the gas plant sits exactly at its size. Compute the finite difference with `+eps` and with `-eps` and compare both with the dual. What does this tell you about reading a single dual?
6. **One line, one interval.** Raise the Ridge–Town limit from 110 to 111 MW at noon only. Check that the cost falls by $\mu = \$120$ and that Ridge output rises by 3 MW while gas falls by 3 MW (why 3?). Then raise the limit in every hour and compare the saving with the sum of the full hours' $\mu$. If they differ, find the hour that explains it.

**Answer check.** Run the next cell after you have tried the tasks. It solves each task's scenario with `build()` (task 4 with its own PTDF-form LP) and asserts the answers; the comments give them in words.

In [ ]:
# Answer check for the six practice tasks
def status(**kw):
    lp = build(**kw)[0]
    return linprog(method="highs", **lp).status            # 0: optimal, 2: infeasible

# 1. 86 MW solves, 85 MW does not. Only 16:00 fails: the town needs 253 MW beyond its nuclear plant, the two gas plants
#    make at most 250 MW, so the Ridge sends at least 3 MW; the direct line then carries 2/3 x 3 + 1/3 x 250 = 85.3 MW.
assert status(network=True, limit_RT=86) == 0 and status(network=True, limit_RT=85) == 2
gas = PLANTS["cc"]["size"] + PLANTS["pk"]["size"]
edge = {t: (2 * max(0, demand[t] - 100 - gas) + min(demand[t] - 100, gas)) / 3 for t in range(HOURS)}
assert max(edge, key=edge.get) == 16 and abs(edge[16] - 256 / 3) < 1e-9

# 2. The nuclear plant on Wind Ridge. At 110 MW, the town's 311 MW at noon must all arrive on its two lines, 110 + 200 MW.
#    The lowest limit that solves is 153 MW, set at 16:00: 353 MW of demand, at most 200 MW on Gas Hub-Town.
#    At 160 MW the noon prices are Town $80 (2 MWh more gas, 1 MWh less from the Ridge), Ridge $0, Gas Hub $40.
PLANTS["nuclear"]["bus"] = "R"
try:
    assert status(network=True, limit_RT=110) == 2
    assert status(network=True, limit_RT=153) == 0 and status(network=True, limit_RT=152) == 2
    N160 = solve(network=True, limit_RT=160); p160 = lmps(N160)
    assert abs(N160["cost"] - 175440) < 1e-6 and np.allclose([p160["T"][12], p160["R"][12], p160["G"][12]], [80, 0, 40], atol=1e-6)
    up = solve(network=True, limit_RT=160, add=("T", 12, eps))
    assert abs((up["cost"] - N160["cost"]) / eps - 80) < 1e-6 and abs((up["x"]["cc_12"] - N160["x"]["cc_12"]) / eps - 2) < 1e-6
finally:
    PLANTS["nuclear"]["bus"] = "T"

# 3. Ridge-Gas Hub at 40 MW: full at 09:00-12:00, 15:00 and 16:00, with Gas Hub $40 > Town $20 > Ridge $0 and mu = $60
#    (Ridge: 20 - 1/3 x 60 = 0; Gas Hub: 20 + 1/3 x 60 = 40). One more MWh in town: half from gas, half from the Ridge.
R40 = solve(network=True, limits={"RG": 40}); p40 = lmps(R40)
full40 = [t for t in range(HOURS) if abs(R40["x"][f"fRG_{t}"]) >= 40 - 1e-6]
assert full40 == [9, 10, 11, 12, 15, 16] and abs(R40["cost"] - 107640) < 1e-6
assert all(np.allclose([p40["G"][t], p40["T"][t], p40["R"][t]], [40, 20, 0], atol=1e-6) for t in full40)
assert abs(-R40["ub_dual"]["cap+_RG_12"] + R40["ub_dual"]["cap-_RG_12"] - 60) < 1e-6
up = solve(network=True, limits={"RG": 40}, add=("T", 12, eps))
assert abs((up["cost"] - R40["cost"]) / eps - 20) < 1e-6 and abs((up["x"]["cc_12"] - R40["x"]["cc_12"]) / eps - 0.5) < 1e-6

# 4. The PTDF form at 110 MW: one balance row per hour, line flows from the plants' outputs (the town, the reference bus,
#    has PTDF 0), the ramp rows of build(). Same cost; LMP_n = LMP_Town - sum_l PTDF[l][n] mu_l gives section 4's prices.
names = [f"{g}_{t}" for t in range(HOURS) for g in PLANTS]; at = {nm: k for k, nm in enumerate(names)}
cost = np.array([PLANTS[nm.split("_")[0]]["cost"] for nm in names], float)
bnds = [(available(g, t), available(g, t)) if g == "nuclear" else (0, available(g, t)) for t in range(HOURS) for g in PLANTS]
Aeq = np.zeros((HOURS, len(names))); Aub, bub = [], []
for t in range(HOURS):
    for g in PLANTS: Aeq[t, at[f"{g}_{t}"]] = 1
    for l in LINES:
        r = np.zeros(len(names))
        for g, p in PLANTS.items(): r[at[f"{g}_{t}"]] = P[l][p["bus"]]
        Aub += [r, -r]; bub += [110.0 if l == "RT" else 200.0] * 2
    r = np.zeros(len(names)); r[at[f"cc_{t}"]] = 1
    if t > 0: r[at[f"cc_{t - 1}"]] = -1
    Aub += [r, -r]; bub += [RAMP, RAMP] if t > 0 else [CC_START + RAMP, RAMP - CC_START]
res4 = linprog(cost, A_ub=np.array(Aub), b_ub=bub, A_eq=Aeq, b_eq=demand, bounds=bnds, method="highs")
assert res4.status == 0 and abs(res4.fun - B110["cost"]) < 1e-6
m4 = res4.ineqlin.marginals.reshape(HOURS, 2 * len(LINES) + 2)
for t in range(HOURS):
    mu4 = {l: -m4[t, 2 * k] + m4[t, 2 * k + 1] for k, l in enumerate(LINES)}
    for n in BUSES:
        assert abs(res4.eqlin.marginals[t] - sum(P[l][n] * mu4[l] for l in LINES) - p110[n][t]) < 1e-6

# 5. The 14:00 corner at 110 MW: one more MWh needs the peaker ($200), one less saves $80; the dual is $80, one end.
up = solve(network=True, limit_RT=110, add=("T", 14, eps)); dn = solve(network=True, limit_RT=110, add=("T", 14, -eps))
assert abs((up["cost"] - B110["cost"]) / eps - 200) < 1e-6 and abs((B110["cost"] - dn["cost"]) / eps - 80) < 1e-6 and abs(p110["T"][14] - 80) < 1e-6

# 6. +1 MW at noon only saves mu = $120: each MW of Ridge power swapped for gas adds 2/3 - 1/3 = 1/3 MW to the line, so the
#    Ridge gives 3 MW more and gas 3 MW less. +1 MW in every hour saves $1,840, less than the full hours' sum of mu, $1,920:
#    at 09:00 the line has room again after a third of a MW, so a whole MW there saves $40, not $120.
lp, cols, _, ubr = build(network=True, limit_RT=110)
for side in ("cap+_RT_12", "cap-_RT_12"): lp["b_ub"][ubr.index(side)] += 1
q = linprog(method="highs", **lp)
ridge = lambda x, t: x[cols[f"wind_{t}"]] + x[cols[f"solar_{t}"]]
assert abs(B110["cost"] - q.fun - 120) < 1e-6
assert abs(ridge(q.x, 12) - B110["x"]["wind_12"] - B110["x"]["solar_12"] - 3) < 1e-6 and abs(q.x[cols["cc_12"]] - B110["x"]["cc_12"] + 3) < 1e-6
sum_mu = sum(-B110["ub_dual"][f"cap+_RT_{t}"] + B110["ub_dual"][f"cap-_RT_{t}"] for t in full)
assert abs(sum_mu - 1920) < 1e-6 and abs(B110["cost"] - solve(network=True, limit_RT=111)["cost"] - 1840) < 1e-6
lp9, _, _, ubr9 = build(network=True, limit_RT=110)
for side in ("cap+_RT_9", "cap-_RT_9"): lp9["b_ub"][ubr9.index(side)] += 1
assert abs(B110["cost"] - linprog(method="highs", **lp9).fun - 40) < 1e-6
print("All six practice answers check out.")